# Transformer lag configuration study

Compares different lag configurations for the Probabilistic Transformer.

## Metrics
- **Point**: MAE, RMSE, MAPE, R2
- **Probabilistic**: PICP, MPIW, PINAW, Interval Score, CRPS
- **Quantile**: Pinball losses (10th, 50th, 90th percentiles)

In [1]:
import os
import sys
import json
import hashlib
import time
import gc
import warnings
from pathlib import Path
from typing import Dict, Any, List, Tuple, Union

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline
from models import ProbabilisticTransformer
from core.trainer import Trainer
from core.evaluator import Evaluator
from transformations import StandardScalingTransformation

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# GPU Setup
try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-03-02 15:17:42.690173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772461062.707273  254489 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772461062.712417  254489 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772461062.725268  254489 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772461062.725285  254489 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772461062.725287  254489 computation_placer.cc:177] computation placer alr

GPUs detected: 1


## Configuration

In [2]:
# Results directory
RESULTS_DIR = Path(project_root) / "results" / "lag_study"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "lag_study_results.json"

# Base model parameters
BASE_MODEL_CONFIG = {
    "d_model": 224,      
    "num_heads": 7,
    "num_layers": 3,
    "ff_dim": 448,
    "dropout": 0.15,
}

# Training settings
TRAIN_SETTINGS = {
    "batch_size": 32,
    "epochs": 30,
    "learning_rate": 7e-4,
    "patience": 7,
    "validation_split": 0.1,
}

OUTPUT_HORIZON = 24
N_RUNS = 3

## Lag generation logic

Functions to generate indices and create dataset sequences

In [3]:
def get_lag_indices(config_name: str) -> List[int]:
    # Parse configuration name and returns a list of lag indices

    import re
    lags = set()
    
    # Recent windows (e.g. recent_24h)
    if "recent_" in config_name:
        match = re.search(r"recent_(\d+)h", config_name)
        if match:
            hours = int(match.group(1))
            for h in range(1, hours + 1):
                lags.add(h)
    
    # Daily (e.g. daily_same_hour_90d)
    if "daily_same_hour_" in config_name:
        match = re.search(r"daily_same_hour_(\d+)d", config_name)
        if match:
            days = int(match.group(1))
            for d in range(1, days + 1):
                lags.add(d * 24)

    # Weekly (e.g. weekly_same_hour_26w)
    if "weekly_same_hour_" in config_name:
        match = re.search(r"weekly_same_hour_(\d+)w", config_name)
        if match:
            weeks = int(match.group(1))
            for w in range(1, weeks + 1):
                lags.add(w * 168)

    # Weekly anchors
    if "weekly" in config_name and "weekly_same_hour" not in config_name:
        match = re.search(r"weekly(\d+)", config_name)
        if match:
            weeks = int(match.group(1))
            for w in range(1, weeks + 1):
                lags.add(w * 168)
    
    # Daily anchors
    if "daily" in config_name and "daily_same_hour" not in config_name:
        match = re.search(r"daily(\d+)", config_name)
        if match:
            days = int(match.group(1))
            for d in range(1, days + 1):
                lags.add(d * 24)

    if not lags:
        raise ValueError(f"Could not get lags from config: {config_name}")
        
    return sorted(list(lags), reverse=True)

def create_lagged_dataset(df: pd.DataFrame, lag_indices: List[int], output_horizon: int) -> Tuple[np.ndarray, np.ndarray]:
    #Creates X (samples, n_lags, features) and y (samples, horizon) dataset

    data = df.values
    n_features = data.shape[1]
    max_lag = max(lag_indices)
    n_samples = len(data) - max_lag - output_horizon + 1
    
    if n_samples <= 0:
        return np.empty((0, len(lag_indices), data.shape[1])), np.empty((0, output_horizon))
    
    t_indices = np.arange(max_lag, len(data) - output_horizon + 1)
    
    lag_arr = np.array(lag_indices)
    X_idx = t_indices[:, None] - lag_arr[None, :]
    
    X = data[X_idx]
    
    y_offsets = np.arange(output_horizon)
    y_idx = t_indices[:, None] + y_offsets[None, :]
    
    y = data[y_idx, 0]
    
    
    valid_mask = ~(np.isnan(X).any(axis=(1,2)) | np.isnan(y).any(axis=1))
    X = X[valid_mask]
    y = y[valid_mask]
    
    return X, y

## Experiment configurations

In [4]:
EXPERIMENTS = []

# Recent windows
for h in [3, 24, 72, 168, 336]:
    EXPERIMENTS.append(f"recent_{h}h")

# Sparse seasonal anchors
EXPERIMENTS.append("daily_same_hour_90d")
EXPERIMENTS.append("daily_same_hour_200d")
EXPERIMENTS.append("weekly_same_hour_26w")
EXPERIMENTS.append("weekly_same_hour_52w")

# Hybrid configurations
EXPERIMENTS.append("recent_24h_plus_daily14")
EXPERIMENTS.append("recent_24h_plus_weekly4")

# 4. Grid search sweep
recent_sweep = [36, 48, 60, 72, 84, 96]
weekly_sweep = [4, 8, 12]

for r in recent_sweep:
    for w in weekly_sweep:
        EXPERIMENTS.append(f"recent_{r}h_plus_weekly{w}")

print(f"Total experiments defined: {len(EXPERIMENTS)}")
print("Sample experiments:", EXPERIMENTS[:5])

Total experiments defined: 29
Sample experiments: ['recent_3h', 'recent_24h', 'recent_72h', 'recent_168h', 'recent_336h']


## Caching

In [5]:
def load_cache() -> Dict[str, Any]:
    if CACHE_FILE.exists():
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except json.JSONDecodeError:
            print("Cache file corrupted, starting fresh.")
            return {}
    return {}

def save_cache(cache: Dict[str, Any]) -> None:
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

## Main experiment loop

In [6]:
# Load data once
print("Loading base data")
data_config = DataConfig(dataset_name="BE_ENTSOE")
pipeline = DataPipeline(data_config)
train_df_raw, val_df_raw, test_df_raw = pipeline.get_data_splits()

# Drop columns with NaN values
cols_with_nan = train_df_raw.columns[train_df_raw.isna().any()].tolist()
if cols_with_nan:
    print(f"Dropping {len(cols_with_nan)} columns with NaN: {cols_with_nan}")
    train_df_raw = train_df_raw.drop(columns=cols_with_nan)
    val_df_raw = val_df_raw.drop(columns=cols_with_nan)
    test_df_raw = test_df_raw.drop(columns=cols_with_nan)

print(f"Features used: {list(train_df_raw.columns)}")

print(f"Train samples: {len(train_df_raw)}")
print(f"Val samples: {len(val_df_raw)}")
print(f"Test samples: {len(test_df_raw)}")

cache = load_cache()

for i, exp_name in enumerate(EXPERIMENTS):
    print(f"\n[{i+1}/{len(EXPERIMENTS)}] Processing: {exp_name}")
    
    if exp_name in cache:
        print("  -> Found in cache. Skipping.")
        continue
    
    # Parse lags
    try:
        lags = get_lag_indices(exp_name)
    except ValueError as e:
        print(f"  -> Error parsing config: {e}")
        continue
        
    n_lags = len(lags)
    print(f"  -> Lags: {n_lags} total (Max lag: {max(lags)}) ")
    
    # Prepare data
    print("  -> Creating matrices...")
    X_train, y_train = create_lagged_dataset(train_df_raw, lags, OUTPUT_HORIZON)
    X_val, y_val = create_lagged_dataset(val_df_raw, lags, OUTPUT_HORIZON)
    X_test, y_test = create_lagged_dataset(test_df_raw, lags, OUTPUT_HORIZON)
    
    if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
        print(f"  -> Skipping: insufficient data (train={len(X_train)}, val={len(X_val)}, test={len(X_test)})")
        continue
    
    # Standard scaling
    transform = StandardScalingTransformation()
    transform.fit(X_train, y_train)
    
    X_train_s, y_train_s = transform.transform(X_train, y_train)
    X_val_s, y_val_s = transform.transform(X_val, y_val)
    X_test_s, y_test_s = transform.transform(X_test, y_test)
    
    # Run repeats
    run_metrics = []
    run_times = []
    
    for run_idx in range(N_RUNS):
        print(f"    Run {run_idx+1}/{N_RUNS}...")
        
        # Cleanup
        tf.keras.backend.clear_session()
        gc.collect()
        
        # Configs
        current_data_config = DataConfig(
            dataset_name="BE_ENTSOE",
            input_window=n_lags, 
            output_horizon=OUTPUT_HORIZON
        )
        
        model_conf = ModelConfig(**BASE_MODEL_CONFIG)
        train_conf = TrainingConfig(**TRAIN_SETTINGS)
        
        exp_conf = ExperimentConfig(
            name=f"{exp_name}_run{run_idx}",
            data_config=current_data_config,
            model_config=model_conf,
            training_config=train_conf,
            head_type="johnson_su"
        )
        
        # Initialize & Train
        model = ProbabilisticTransformer(exp_conf)
        trainer = Trainer(exp_conf)
        
        start_time = time.time()
        history = trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s)
        fit_time = time.time() - start_time
        
        # Evaluate
        evaluator = Evaluator(model, transform)
        metrics = evaluator.evaluate(X_test_s, y_test) # user raw y_test for metrics
        
        run_metrics.append(metrics)
        run_times.append(fit_time)
        
        del model, trainer, evaluator
    
    # Aggregate results
    avg_metrics = {}
    std_metrics = {}
    
    keys = run_metrics[0].keys()
    for k in keys:
        values = [m[k] for m in run_metrics]
        avg_metrics[k] = float(np.mean(values))
        std_metrics[k] = float(np.std(values))
    
    result_entry = {
        "experiment": exp_name,
        "lags_count": n_lags,
        "metrics_avg": avg_metrics,
        "metrics_std": std_metrics,
        "avg_fit_time": float(np.mean(run_times)),
        "runs": N_RUNS
    }
    
    # Save update
    cache[exp_name] = result_entry
    save_cache(cache)
    
    extra = ""
    if avg_metrics.get('CRPS') is not None: extra += f", CRPS={avg_metrics['CRPS']:.4f}"
    if avg_metrics.get('PICP') is not None: extra += f", PICP={avg_metrics['PICP']:.2%}"
    if avg_metrics.get('R2') is not None and not np.isnan(avg_metrics.get('R2')): extra += f", R2={avg_metrics['R2']:.4f}"
    print(f"  -> Done. MAE={avg_metrics.get('MAE', 0):.4f}, RMSE={avg_metrics.get('RMSE', 0):.4f}{extra}")

Loading base data
Dropping 5 columns with NaN: ['FR_Generation_Forecast', 'BE_Wind_Forecast', 'BE_Solar_Forecast', 'Flow_BE_NL', 'BE_Generation_Actual']
Features used: ['Prices', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'DayOfYear', 'WeekOfYear', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'NL_Prices', 'FR_Prices', 'FR_Load_Forecast', 'BE_Load_Actual', 'BE_Load_Forecast', 'Price_Spread_FR', 'Flow_BE_FR', 'BE_Load_Imbalance', 'Price_Spread_NL', 'Flow_BE_DE']
Train samples: 19700
Val samples: 2188
Test samples: 13278

[1/29] Processing: recent_3h
  -> Found in cache. Skipping.

[2/29] Processing: recent_24h
  -> Found in cache. Skipping.

[3/29] Processing: recent_72h
  -> Found in cache. Skipping.

[4/29] Processing: recent_168h
  -> Found in cache. Skipping.

[5/29] Processing: recent_336h
  -> Found in cache. Skipping.

[6/29] Processing: daily_same_hour_90d
  -> Found in cache. Skipping.

[7/29] Processing: daily_same_hour_200d
  -> Lags: 200

## Results analysis

In [7]:
METRIC_COLS = ["MAE", "RMSE", "MAPE", "R2", "PICP", "MPIW", "PINAW", "IntervalScore", "CRPS", "Pinball_10", "Pinball_50", "Pinball_90", "Avg_Pinball"]
m_avg = "metrics_avg"

results_list = []
for k, v in cache.items():
    row = {
        "Configuration": k,
        "Lags": v["lags_count"],
        "Time(s)": v["avg_fit_time"]
    }
    for col in METRIC_COLS:
        val = v[m_avg].get(col)
        if val is not None and (not isinstance(val, float) or not np.isnan(val)):
            row[col] = val
    results_list.append(row)

df_res = pd.DataFrame(results_list)
if not df_res.empty:
    display_cols = [c for c in METRIC_COLS if c in df_res.columns]
    df_res = df_res[["Configuration", "Lags"] + display_cols + ["Time(s)"]]
    df_res = df_res.sort_values("MAE")
    display(df_res)
    
    df_res.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
else:
    print("No results to display.")

,Configuration,Lags,MAE,RMSE,Time(s)
11,recent_48h_plus_weekly8,56,17.100547,24.205063,115.295465
15,recent_72h_plus_weekly8,80,18.376989,25.165530,136.101630
4,recent_336h,336,18.610549,25.150685,384.122162
2,recent_72h,72,18.622386,24.791405,138.607546
1,recent_24h,24,18.813955,25.086154,100.028378
3,recent_168h,168,18.825557,25.280858,206.032883
6,recent_24h_plus_daily14,37,18.912816,25.503852,128.670593
17,recent_84h_plus_weekly8,92,18.933922,25.610225,135.635341
9,recent_36h_plus_weekly8,44,19.167160,26.389366,109.195394
0,recent_3h,3,19.527222,26.254623,106.759260
